# TP5 chatbot RAG y evaluación de embeddings

# **Aclaración: El tp 5 comienza luego del tp4.**

# TP4 Chatbots basados en recuperación de la información


# Alumno: Moises Lobayza

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

# Librerías

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 57.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# Actividades

### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

**Respuesta:**

## Aplicación elegida

La aplicación elegida es un chatbot para un **consultorio médico privado**.

El **objetivo** del chatbot es responder consultas frecuentes de pacientes relacionadas con turnos, horarios de atención, ubicación del consultorio, formas de pago, cancelaciones, reprogramaciones, estudios previos, recetas, certificados y consultas administrativas generales.

El chatbot no reemplaza la atención médica profesional ni realiza diagnósticos. **Su función principal es brindar información administrativa y orientar al paciente para que pueda comunicarse correctamente con el consultorio o solicitar un turno**.

In [ ]:
data = [
    {
        "pregunta": "¿Cómo puedo sacar un turno?",
        "respuesta": "Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta."
    },
    {
        "pregunta": "¿Cuáles son los horarios de atención?",
        "respuesta": "El consultorio atiende de lunes a viernes de 9:00 a 18:00 hs. Los horarios pueden variar según la disponibilidad del profesional."
    },
    {
        "pregunta": "¿Dónde queda el consultorio?",
        "respuesta": "El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno."
    },
    {
        "pregunta": "¿Atienden por obra social?",
        "respuesta": "Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno."
    },
    {
        "pregunta": "¿Puedo atenderme de forma particular?",
        "respuesta": "Sí, también se atienden pacientes particulares. El valor de la consulta se informa al momento de solicitar el turno."
    },
    {
        "pregunta": "¿Cómo hago para cancelar un turno?",
        "respuesta": "Para cancelar un turno, escribí al WhatsApp del consultorio con tu nombre, apellido y fecha del turno que querés cancelar."
    },
    {
        "pregunta": "¿Puedo reprogramar mi turno?",
        "respuesta": "Sí, podés reprogramar tu turno escribiendo al consultorio. Se te ofrecerán nuevas opciones según la disponibilidad."
    },
    {
        "pregunta": "¿Qué datos necesito para pedir un turno?",
        "respuesta": "Para pedir un turno necesitás informar nombre, apellido, DNI, obra social si tenés, motivo de consulta y disponibilidad horaria."
    },
    {
        "pregunta": "¿Atienden urgencias?",
        "respuesta": "El consultorio no atiende urgencias. En caso de emergencia médica, dirigite a una guardia o llamá al servicio de emergencias correspondiente."
    },
    {
        "pregunta": "¿Puedo pedir una receta por WhatsApp?",
        "respuesta": "Las recetas deben ser evaluadas por el profesional. En algunos casos pueden solicitarse por WhatsApp si ya sos paciente del consultorio."
    },
    {
        "pregunta": "¿Hacen certificados médicos?",
        "respuesta": "Sí, el profesional puede realizar certificados médicos cuando corresponda, luego de una evaluación en consulta."
    },
    {
        "pregunta": "¿Cuánto dura una consulta?",
        "respuesta": "La duración aproximada de una consulta es de 20 a 30 minutos, aunque puede variar según el motivo de atención."
    },
    {
        "pregunta": "¿Qué pasa si llego tarde al turno?",
        "respuesta": "Si llegás tarde, el profesional intentará atenderte según la disponibilidad del día. En algunos casos puede ser necesario reprogramar."
    },
    {
        "pregunta": "¿Debo llevar estudios anteriores?",
        "respuesta": "Sí, si tenés estudios previos, análisis, recetas o informes médicos, es recomendable llevarlos a la consulta."
    },
    {
        "pregunta": "¿Puedo ir acompañado a la consulta?",
        "respuesta": "Sí, podés asistir acompañado si lo necesitás. En algunos casos el ingreso del acompañante puede depender del espacio disponible."
    },
    {
        "pregunta": "¿Atienden niños?",
        "respuesta": "La atención a niños depende de la especialidad del profesional. Consultá previamente indicando la edad del paciente."
    },
    {
        "pregunta": "¿Cuáles son las formas de pago?",
        "respuesta": "El consultorio acepta efectivo y transferencias. Otros medios de pago pueden consultarse al momento de confirmar el turno."
    },
    {
        "pregunta": "¿Entregan factura?",
        "respuesta": "Sí, el consultorio puede emitir comprobante o factura por la atención realizada."
    },
    {
        "pregunta": "¿Puedo consultar resultados de estudios por WhatsApp?",
        "respuesta": "Los resultados deben ser evaluados por el profesional. Podés enviarlos por WhatsApp, pero la interpretación se realizará en consulta o según indicación médica."
    },
    {
        "pregunta": "¿Cómo confirmo mi turno?",
        "respuesta": "Para confirmar tu turno, respondé el mensaje de confirmación enviado por el consultorio indicando que vas a asistir."
    }
]

df = pd.DataFrame(data)

df

,pregunta,respuesta
0,¿Cómo puedo sacar un turno?,Podés solicitar un turno escribiendo por Whats...
1,¿Cuáles son los horarios de atención?,El consultorio atiende de lunes a viernes de 9...
2,¿Dónde queda el consultorio?,El consultorio se encuentra en Córdoba Capital...
3,¿Atienden por obra social?,"Sí, el consultorio trabaja con algunas obras s..."
4,¿Puedo atenderme de forma particular?,"Sí, también se atienden pacientes particulares..."
5,¿Cómo hago para cancelar un turno?,"Para cancelar un turno, escribí al WhatsApp de..."
6,¿Puedo reprogramar mi turno?,"Sí, podés reprogramar tu turno escribiendo al ..."
7,¿Qué datos necesito para pedir un turno?,"Para pedir un turno necesitás informar nombre,..."
8,¿Atienden urgencias?,El consultorio no atiende urgencias. En caso d...
9,¿Puedo pedir una receta por WhatsApp?,Las recetas deben ser evaluadas por el profesi...


# Limpieza general antes de aplicar los modelos.

Para simplificar el trabajo se utilizó el mismo preprocesamiento en ambos modelos. Sin embargo, **TF-IDF depende más de la limpieza porque trabaja con palabras y frecuencias, mientras que los embeddings pueden trabajar mejor con texto natural y no siempre requieren una limpieza tan fuerte.**

No se eliminaron stop words porque el dataset contiene preguntas cortas y administrativas, donde algunas palabras funcionales pueden aportar contexto. Se mantuvo una limpieza básica para estandarizar el texto sin perder información relevante.


In [ ]:
def limpiar_texto(texto):
    texto = str(texto).lower()

    # Quitar tildes
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Quitar signos de puntuación y caracteres especiales
    texto = re.sub(r'[^a-zñ\s]', '', texto)

    # Quitar espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto


# Aplicar limpieza a preguntas y respuestas
df["pregunta_limpia"] = df["pregunta"].apply(limpiar_texto)
df["respuesta_limpia"] = df["respuesta"].apply(limpiar_texto)

# Verificar resultado
df[["pregunta", "pregunta_limpia", "respuesta", "respuesta_limpia"]].head()

,pregunta,pregunta_limpia,respuesta,respuesta_limpia
0,¿Cómo puedo sacar un turno?,como puedo sacar un turno,Podés solicitar un turno escribiendo por Whats...,podes solicitar un turno escribiendo por whats...
1,¿Cuáles son los horarios de atención?,cuales son los horarios de atencion,El consultorio atiende de lunes a viernes de 9...,el consultorio atiende de lunes a viernes de a...
2,¿Dónde queda el consultorio?,donde queda el consultorio,El consultorio se encuentra en Córdoba Capital...,el consultorio se encuentra en cordoba capital...
3,¿Atienden por obra social?,atienden por obra social,"Sí, el consultorio trabaja con algunas obras s...",si el consultorio trabaja con algunas obras so...
4,¿Puedo atenderme de forma particular?,puedo atenderme de forma particular,"Sí, también se atienden pacientes particulares...",si tambien se atienden pacientes particulares ...


# Funciones de uso común

En esta sección se definieron funciones generales que pueden ser reutilizadas tanto por el chatbot basado en **TF-IDF** como por el chatbot basado en **embeddings**.

**La función `chatbot_retrieval()` contiene la lógica principal del chatbot.** Recibe una consulta del usuario, la vectoriza con el método correspondiente, calcula la similitud del coseno contra las preguntas del dataset y devuelve la respuesta asociada a la pregunta más similar. También aplica un umbral mínimo de similitud para evitar responder cuando no encuentra una coincidencia adecuada.

**La función `preparar_matriz_preguntas()` se utiliza para convertir las preguntas limpias del dataset en una matriz de vectores.** Esta función es general porque puede recibir distintos métodos de vectorización, por ejemplo TF-IDF o embeddings.

In [ ]:
def chatbot_retrieval(consulta_usuario, matriz_referencia, funcion_vectorizar, umbral):
    # Vectorizar la consulta según el modelo elegido
    consulta_vectorizada = funcion_vectorizar(consulta_usuario)

    # Calcular similitud del coseno
    similitudes = cosine_similarity(consulta_vectorizada, matriz_referencia)

    # Buscar la pregunta más similar
    indice_mas_similar = similitudes.argmax()

    # Obtener mejor similitud
    mejor_similitud = similitudes[0, indice_mas_similar]

    # Aplicar umbral mínimo
    if mejor_similitud < umbral:
        return "No encontré una respuesta adecuada. Te recomiendo comunicarte directamente con el consultorio."

    # Devolver respuesta original asociada
    return df.iloc[indice_mas_similar]["respuesta"]

In [ ]:
def preparar_matriz_preguntas(preguntas_limpias, funcion_vectorizar_dataset):
    return funcion_vectorizar_dataset(preguntas_limpias)

# 2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

**En este bloque se construye el chatbot basado en TF-IDF.**

Primero se crea el vectorizador `TfidfVectorizer`, que permite transformar texto en vectores numéricos según la importancia de las palabras.

Luego, la función `vectorizar_dataset_tfidf()` aplica `fit_transform()` sobre las preguntas limpias del dataset. Esto significa que el vectorizador aprende el vocabulario de las preguntas y genera la matriz `matriz_tfidf`, que contiene los vectores de referencia.

Después, la función `vectorizar_tfidf()` se utiliza para transformar una **consulta** nueva del usuario usando el mismo vectorizador ya ajustado.

Finalmente, `chatbot_tfidf()` llama a la función general `chatbot_retrieval()`, pasándole la matriz TF-IDF, la función de vectorización correspondiente y un umbral mínimo de similitud de 0.2.

In [ ]:
# Crear vectorizador TF-IDF
vectorizador_tfidf = TfidfVectorizer()

def vectorizar_dataset_tfidf(preguntas_limpias):
    return vectorizador_tfidf.fit_transform(preguntas_limpias)

matriz_tfidf = preparar_matriz_preguntas(
    df["pregunta_limpia"],
    vectorizar_dataset_tfidf
)

In [ ]:
def vectorizar_tfidf(consulta_usuario):
    consulta_limpia = limpiar_texto(consulta_usuario)
    return vectorizador_tfidf.transform([consulta_limpia])

In [ ]:
def chatbot_tfidf(consulta_usuario):
    return chatbot_retrieval(
        consulta_usuario,
        matriz_tfidf,
        vectorizar_tfidf,
        umbral=0.2
    )

# 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado eligió.

**En este bloque se construye el chatbot basado en embeddings.**

Primero se carga el modelo preentrenado `paraphrase-multilingual-MiniLM-L12-v2`, que permite **transformar oraciones completas en vectores semánticos. Se eligió este modelo porque es multilingüe, funciona con español y está pensado para comparar similitud entre frases**.

Luego, la función `vectorizar_dataset_embeddings()` convierte las preguntas limpias del dataset en embeddings. Esos vectores se guardan en `embeddings_preguntas`, que funciona como matriz de referencia.

Después, la función `vectorizar_embeddings()` transforma cada consulta nueva del usuario en un embedding usando el mismo modelo.

Finalmente, `chatbot_embeddings()` llama a la función general `chatbot_retrieval()`, pasándole los embeddings de las preguntas, la función de vectorización correspondiente y un umbral mínimo de similitud de 0.45.

In [ ]:
modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def vectorizar_dataset_embeddings(preguntas_limpias):
    return modelo_embeddings.encode(preguntas_limpias.tolist())

embeddings_preguntas = preparar_matriz_preguntas(
    df["pregunta_limpia"],
    vectorizar_dataset_embeddings
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def vectorizar_embeddings(consulta_usuario):
    consulta_limpia = limpiar_texto(consulta_usuario)
    return modelo_embeddings.encode([consulta_limpia])

In [ ]:
def chatbot_embeddings(consulta_usuario):
    return chatbot_retrieval(
        consulta_usuario,
        embeddings_preguntas,
        vectorizar_embeddings,
        umbral=0.45
    )

# 4) Muestra ambos chatbots funcionando (1 punto)
Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

# Consultas TF-IDF.

In [ ]:
print(chatbot_tfidf("quiero sacar un turno"))
print(chatbot_tfidf("donde queda el consultorio"))
print(chatbot_tfidf("donde queda el consultorio?????"))
print(chatbot_tfidf("Hola, atienden con obra social???"))
print(chatbot_tfidf("necesito cancelar mi cita"))
print(chatbot_tfidf("tengo una urgencia medica"))
print(chatbot_tfidf("ajsjhshshhshshs"))
print(chatbot_tfidf("puedo pagar por transferencia"))
print(chatbot_tfidf("tengo que llevar estudios anteriores"))
print(chatbot_tfidf("puedo pedir una receta por whatsapp"))

Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno.
Sí, podés reprogramar tu turno escribiendo al consultorio. Se te ofrecerán nuevas opciones según la disponibilidad.
La duración aproximada de una consulta es de 20 a 30 minutos, aunque puede variar según el motivo de atención.
No encontré una respuesta adecuada. Te recomiendo comunicarte directamente con el consultorio.
Las recetas deben ser evaluadas por el profesional. En algunos casos pueden solicitarse por WhatsApp si ya sos paciente del consultorio.
Sí, si tenés estudios previos, análisis, recetas o informes médicos, es

# Análisis de las respuestas del modelo TF-IDF.

In [ ]:
consultas_prueba = [
    "quiero sacar un turno",
    "donde queda el consultorio",
    "donde queda el consultorio?????",
    "Hola, atienden con obra social???",
    "necesito cancelar mi cita",
    "tengo una urgencia medica",
    "ajsjhshshhshshs",
    "puedo pagar por transferencia",
    "tengo que llevar estudios anteriores",
    "puedo pedir una receta por whatsapp"
]

respuestas_tfidf = [chatbot_tfidf(consulta) for consulta in consultas_prueba]

evaluacion_tfidf = [
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Incorrecta",
    "Incorrecta",
    "Correcta",
    "Incorrecta",
    "Correcta",
    "Correcta"
]

observaciones_tfidf = [
    "Detectó correctamente la intención de solicitar turno.",
    "Respondió correctamente sobre la ubicación.",
    "La limpieza permitió ignorar los signos de puntuación.",
    "Detectó correctamente la consulta sobre obra social.",
    "Confundió cancelar con reprogramar.",
    "No detectó correctamente que era una urgencia.",
    "Correctamente no encontró una respuesta adecuada.",
    "Confundió la consulta de pago con recetas.",
    "Respondió correctamente sobre estudios previos.",
    "Respondió correctamente sobre recetas por WhatsApp."
]

df_resultados_tfidf = pd.DataFrame({
    "Consulta": consultas_prueba,
    "Respuesta_tfidf": respuestas_tfidf,
    "Evaluacion": evaluacion_tfidf,
    "Observacion": observaciones_tfidf
})

df_resultados_tfidf

,Consulta,Respuesta_tfidf,Evaluacion,Observacion
0,quiero sacar un turno,Podés solicitar un turno escribiendo por Whats...,Correcta,Detectó correctamente la intención de solicita...
1,donde queda el consultorio,El consultorio se encuentra en Córdoba Capital...,Correcta,Respondió correctamente sobre la ubicación.
2,donde queda el consultorio?????,El consultorio se encuentra en Córdoba Capital...,Correcta,La limpieza permitió ignorar los signos de pun...
3,"Hola, atienden con obra social???","Sí, el consultorio trabaja con algunas obras s...",Correcta,Detectó correctamente la consulta sobre obra s...
4,necesito cancelar mi cita,"Sí, podés reprogramar tu turno escribiendo al ...",Incorrecta,Confundió cancelar con reprogramar.
5,tengo una urgencia medica,La duración aproximada de una consulta es de 2...,Incorrecta,No detectó correctamente que era una urgencia.
6,ajsjhshshhshshs,No encontré una respuesta adecuada. Te recomie...,Correcta,Correctamente no encontró una respuesta adecuada.
7,puedo pagar por transferencia,Las recetas deben ser evaluadas por el profesi...,Incorrecta,Confundió la consulta de pago con recetas.
8,tengo que llevar estudios anteriores,"Sí, si tenés estudios previos, análisis, recet...",Correcta,Respondió correctamente sobre estudios previos.
9,puedo pedir una receta por whatsapp,Las recetas deben ser evaluadas por el profesi...,Correcta,Respondió correctamente sobre recetas por What...


# Conteo final modelo TF-IDF

In [ ]:
df_resultados_tfidf["Evaluacion"].value_counts()

,count
Evaluacion,
Correcta,7
Incorrecta,3


# Conclusión del modelo TF-IDF

El chatbot basado en TF-IDF respondió correctamente en 7 de 10 consultas. Funcionó bien cuando la pregunta del usuario tenía palabras similares a las preguntas del dataset, como en los casos de turnos, ubicación, obra social, estudios anteriores y recetas.

Sin embargo, presentó **errores** cuando el usuario usó palabras distintas o sinónimos, por ejemplo “cita” en lugar de “turno”, o cuando la consulta requería entender mejor el significado. **Esto muestra que TF-IDF depende mucho de la coincidencia de palabras y tiene limitaciones para interpretar el contexto semántico.**

# Consultas Embeddings.

In [ ]:
print(chatbot_embeddings("quiero sacar un turno"))
print(chatbot_embeddings("donde queda el consultorio"))
print(chatbot_embeddings("donde queda el consultorio?????"))
print(chatbot_embeddings("Hola, atienden con obra social???"))
print(chatbot_embeddings("necesito cancelar mi cita"))
print(chatbot_embeddings("tengo una urgencia medica"))
print(chatbot_embeddings("ajsjhshshhshshs"))
print(chatbot_embeddings("puedo pagar por transferencia"))
print(chatbot_embeddings("tengo que llevar estudios anteriores"))
print(chatbot_embeddings("puedo pedir una receta por whatsapp"))

Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.
Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno.
Para cancelar un turno, escribí al WhatsApp del consultorio con tu nombre, apellido y fecha del turno que querés cancelar.
El consultorio no atiende urgencias. En caso de emergencia médica, dirigite a una guardia o llamá al servicio de emergencias correspondiente.
La atención a niños depende de la especialidad del profesional. Consultá previamente indicando la edad del paciente.
Sí, el consultorio puede emitir comprobante o factura por la atención realizada.
Sí, si tenés estudios previos, análisis, recetas o informes médicos

# Análisis de las respuestas del modelo Embeddings.

In [ ]:
respuestas_embeddings = [chatbot_embeddings(consulta) for consulta in consultas_prueba]

evaluacion_embeddings = [
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Correcta",
    "Incorrecta",
    "Incorrecta",
    "Correcta",
    "Correcta"
]

observaciones_embeddings = [
    "Detectó correctamente la intención de solicitar turno.",
    "Respondió correctamente sobre la ubicación.",
    "La limpieza permitió ignorar los signos de puntuación.",
    "Detectó correctamente la consulta sobre obra social.",
    "Detectó correctamente la intención de cancelar un turno.",
    "Respondió correctamente ante una urgencia médica.",
    "No rechazó la consulta sin sentido y devolvió una respuesta incorrecta.",
    "Confundió la consulta sobre forma de pago con emisión de factura.",
    "Respondió correctamente sobre estudios previos.",
    "Respondió correctamente sobre recetas por WhatsApp."
]

df_resultados_embeddings = pd.DataFrame({
    "consulta": consultas_prueba,
    "respuesta_embeddings": respuestas_embeddings,
    "evaluacion": evaluacion_embeddings,
    "observacion": observaciones_embeddings
})

df_resultados_embeddings

,consulta,respuesta_embeddings,evaluacion,observacion
0,quiero sacar un turno,Podés solicitar un turno escribiendo por Whats...,Correcta,Detectó correctamente la intención de solicita...
1,donde queda el consultorio,El consultorio se encuentra en Córdoba Capital...,Correcta,Respondió correctamente sobre la ubicación.
2,donde queda el consultorio?????,El consultorio se encuentra en Córdoba Capital...,Correcta,La limpieza permitió ignorar los signos de pun...
3,"Hola, atienden con obra social???","Sí, el consultorio trabaja con algunas obras s...",Correcta,Detectó correctamente la consulta sobre obra s...
4,necesito cancelar mi cita,"Para cancelar un turno, escribí al WhatsApp de...",Correcta,Detectó correctamente la intención de cancelar...
5,tengo una urgencia medica,El consultorio no atiende urgencias. En caso d...,Correcta,Respondió correctamente ante una urgencia médica.
6,ajsjhshshhshshs,La atención a niños depende de la especialidad...,Incorrecta,No rechazó la consulta sin sentido y devolvió ...
7,puedo pagar por transferencia,"Sí, el consultorio puede emitir comprobante o ...",Incorrecta,Confundió la consulta sobre forma de pago con ...
8,tengo que llevar estudios anteriores,"Sí, si tenés estudios previos, análisis, recet...",Correcta,Respondió correctamente sobre estudios previos.
9,puedo pedir una receta por whatsapp,Las recetas deben ser evaluadas por el profesi...,Correcta,Respondió correctamente sobre recetas por What...


# Conteo final modelo Embeddings.

In [ ]:
df_resultados_embeddings["evaluacion"].value_counts()

,count
evaluacion,
Correcta,8
Incorrecta,2


### Conclusión del modelo con embeddings

El chatbot basado en embeddings respondió correctamente en 8 de 10 consultas. En general tuvo mejor desempeño que TF-IDF porque logró interpretar mejor consultas formuladas con palabras distintas, como “necesito cancelar mi cita” o “tengo una urgencia médica”.

De todos modos, también cometió errores: ante una consulta sin sentido devolvió una respuesta incorrecta, y confundió una pregunta sobre transferencia con una respuesta sobre factura. **Esto muestra que los embeddings capturan mejor el significado, pero también necesitan un buen umbral de similitud y una base de respuestas bien diseñada.**

# 5) Añade tus conclusiones de todo lo realizado (2 punto)
Resalte e indique en cuáles respuestas falla o tiene problemas.

Cuáles preguntas confunde.

Compare los resultados de los chatbots.

## 5) Conclusiones finales

En este trabajo se construyeron dos chatbots de recuperación de información para un consultorio médico privado: uno con TF-IDF y otro con embeddings. Ambos usaron limpieza de texto y similitud del coseno para buscar la pregunta más parecida del dataset.

El chatbot con TF-IDF respondió correctamente 7 de 10 consultas. Funcionó bien cuando había coincidencia directa de palabras, pero falló con sinónimos o frases menos similares. Por ejemplo, confundió “cancelar mi cita” con reprogramar, “urgencia médica” con duración de consulta y “pagar por transferencia” con recetas.

El chatbot con embeddings respondió correctamente 8 de 10 consultas. Tuvo mejor desempeño porque interpretó mejor el significado de frases distintas, como “cancelar mi cita” y “urgencia médica”. Sin embargo, también falló ante una consulta sin sentido y confundió la pregunta sobre transferencia con factura.

En comparación, TF-IDF es más simple y fácil de interpretar, pero depende mucho de las palabras exactas. **Los embeddings resultaron más adecuados para este caso, aunque también requieren ajustar el umbral de similitud y mejorar la base de respuestas.**

## Referencias

- Scikit-learn. Documentación oficial de `TfidfVectorizer`. Se utilizó como referencia para la vectorización TF-IDF de las preguntas del dataset.

- Scikit-learn. Documentación oficial de `cosine_similarity`. Se utilizó como referencia para calcular la similitud del coseno entre la consulta del usuario y las preguntas vectorizadas.

- Sentence Transformers. Documentación sobre modelos preentrenados. Se utilizó como referencia para el uso de modelos de embeddings preentrenados.

- Hugging Face. Modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Se utilizó como embedding preentrenado para representar oraciones completas y comparar similitud semántica.

- Apuntes y notebook de clase sobre procesamiento de lenguaje natural, TF-IDF, embeddings y similitud entre textos.

- LLM usados para consulta: Gpt, Gemini.

- Github: https://github.com/moiseslobayza/procesamiento_del_habla/blob/main/TP4_chatbot_retrieval_template_parte_1_moiseslobayza.ipynb

# TP5 chatbot RAG y evaluación de embeddings

# Introdución:

En el trabajo anterior, el chatbot respondía de una forma más directa: buscaba en la base de conocimiento la pregunta más parecida a la consulta del usuario y devolvía la respuesta asociada. Es decir, **funcionaba principalmente como un sistema de recuperación de información.**

**pregunta nueva → busca la más parecida → devuelve la respuesta guardada**

En este trabajo, el chatbot incorpora un LLM. Ahora primero recupera el contexto más relevante usando embeddings y una base vectorial, y luego le pasa ese contexto al modelo de lenguaje para que genere una respuesta. Por eso el sistema se acerca más a un **esquema RAG, donde se combina recuperación de información con generación de texto.**

**pregunta nueva → embedding de la pregunta → búsqueda en base vectorial
→ recupera contexto relacionado → LLM redacta una respuesta usando ese contexto**


**a)** Creación del conjunto de datos de evaluación.

Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.



**Respuesta:**

Para evaluar el chatbot, además del dataset original creado en el TP4, se construyó un nuevo conjunto de datos de prueba. Este dataset mantiene la misma lógica del trabajo anterior: preguntas frecuentes y respuestas esperadas para un chatbot de atención de un consultorio médico privado.

El dataset original será utilizado como base de conocimiento, mientras que este nuevo dataset será usado para probar si el chatbot puede recuperar información correcta ante preguntas formuladas de distintas maneras.

**Dataset de evaluación:**

In [ ]:
preguntas_eval = [
    "Quiero reservar una consulta médica",
    "¿En qué horario atiende el consultorio?",
    "¿Me pueden pasar la ubicación del consultorio?",
    "¿Trabajan con obras sociales?",
    "¿Puedo pagar la consulta de manera particular?",
    "Necesito cancelar mi turno",
    "Quiero cambiar el día de mi cita",
    "¿Qué información tengo que enviar para pedir turno?",
    "Tengo una emergencia, ¿puedo ir al consultorio?",
    "¿Puedo pedir una receta por WhatsApp?",
    "¿Hacen certificados médicos?",
    "¿Cuánto dura aproximadamente la consulta?",
    "¿Puedo abonar con transferencia?",
    "¿Entregan factura por la atención?",
    "¿Tengo que llevar estudios previos?",
    "¿Atienden niños?",
    "¿Atienden adultos mayores?",
    "¿Puedo ir sin turno previo?",
    "¿Cómo confirmo mi turno?",
    "¿Qué hago si llego tarde?"
]

respuestas_eval = [
    "Podés solicitar un turno escribiendo por WhatsApp al consultorio e indicando tu nombre, apellido, obra social y motivo de consulta.",
    "El consultorio atiende de lunes a viernes de 9:00 a 18:00 hs. Los horarios pueden variar según la disponibilidad del profesional.",
    "El consultorio se encuentra en Córdoba Capital. La dirección exacta se informa al confirmar el turno.",
    "Sí, el consultorio trabaja con algunas obras sociales. Para confirmar cobertura, indicá cuál es tu obra social al solicitar el turno.",
    "Sí, también se atienden pacientes particulares. El valor de la consulta se informa al momento de solicitar el turno.",
    "Para cancelar un turno, escribí al WhatsApp del consultorio con tu nombre, apellido y fecha del turno que querés cancelar.",
    "Sí, podés reprogramar tu turno escribiendo al consultorio. Se te ofrecerán nuevas opciones según la disponibilidad.",
    "Para pedir un turno necesitás informar nombre, apellido, DNI, obra social si tenés, motivo de consulta y disponibilidad horaria.",
    "El consultorio no atiende urgencias. En caso de emergencia médica, dirigite a una guardia o llamá al servicio de emergencias correspondiente.",
    "Sí, podés solicitar recetas por WhatsApp si ya sos paciente del consultorio. El profesional evaluará si corresponde emitirla.",
    "Sí, el profesional puede realizar certificados médicos cuando corresponda, luego de la evaluación del paciente.",
    "La duración aproximada de la consulta depende del motivo de atención, pero generalmente puede durar entre 20 y 40 minutos.",
    "Sí, se puede abonar por transferencia. Los datos de pago se informan al momento de confirmar el turno.",
    "Sí, se puede solicitar factura por la atención brindada en el consultorio.",
    "Sí, es recomendable llevar estudios previos, recetas, análisis o informes relacionados con el motivo de consulta.",
    "La atención a niños depende de la especialidad del profesional. Se recomienda consultar previamente por WhatsApp.",
    "Sí, el consultorio atiende adultos mayores. Se recomienda informar si necesitan asistencia especial.",
    "No se recomienda asistir sin turno previo. Para una mejor organización, es necesario solicitar turno antes de concurrir.",
    "El turno se confirma por WhatsApp una vez acordado el día, horario y los datos necesarios del paciente.",
    "Si llegás tarde, debés avisar al consultorio por WhatsApp. Según la demora y la agenda del día, el turno puede mantenerse o reprogramarse."
]

In [ ]:
df_eval = pd.DataFrame({
    "pregunta": preguntas_eval,
    "respuesta_esperada": respuestas_eval
})

df_eval

,pregunta,respuesta_esperada
0,Quiero reservar una consulta médica,Podés solicitar un turno escribiendo por Whats...
1,¿En qué horario atiende el consultorio?,El consultorio atiende de lunes a viernes de 9...
2,¿Me pueden pasar la ubicación del consultorio?,El consultorio se encuentra en Córdoba Capital...
3,¿Trabajan con obras sociales?,"Sí, el consultorio trabaja con algunas obras s..."
4,¿Puedo pagar la consulta de manera particular?,"Sí, también se atienden pacientes particulares..."
5,Necesito cancelar mi turno,"Para cancelar un turno, escribí al WhatsApp de..."
6,Quiero cambiar el día de mi cita,"Sí, podés reprogramar tu turno escribiendo al ..."
7,¿Qué información tengo que enviar para pedir t...,"Para pedir un turno necesitás informar nombre,..."
8,"Tengo una emergencia, ¿puedo ir al consultorio?",El consultorio no atiende urgencias. En caso d...
9,¿Puedo pedir una receta por WhatsApp?,"Sí, podés solicitar recetas por WhatsApp si ya..."


In [ ]:
print("Cantidad de preguntas de evaluación:", len(df_eval))
df_eval.head()

Cantidad de preguntas de evaluación: 20


,pregunta,respuesta_esperada
0,Quiero reservar una consulta médica,Podés solicitar un turno escribiendo por Whats...
1,¿En qué horario atiende el consultorio?,El consultorio atiende de lunes a viernes de 9...
2,¿Me pueden pasar la ubicación del consultorio?,El consultorio se encuentra en Córdoba Capital...
3,¿Trabajan con obras sociales?,"Sí, el consultorio trabaja con algunas obras s..."
4,¿Puedo pagar la consulta de manera particular?,"Sí, también se atienden pacientes particulares..."


**b)** Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.




**Respuesta:**

Para este trabajo elegí un modelo LLM de HuggingFace y dos modelos de embeddings para comparar el rendimiento del chatbot.

Como modelo LLM usaré **`google/flan-t5-small`**. Lo elegí porque es un modelo liviano, instructivo y adecuado para realizar pruebas en notebooks. En este trabajo no se busca entrenar un modelo desde cero, sino usar un modelo generativo para redactar una respuesta a partir del contexto recuperado por la base vectorial. Además, al ser un modelo más liviano, permite probar el funcionamiento general del chatbot con menor tiempo de carga y menor consumo de recursos.

Como modelos de embeddings usaré:

1. **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**
2. **`intfloat/multilingual-e5-small`**

Elegí estos modelos porque ambos son multilingües y permiten representar textos en forma de vectores. Esto es importante porque el dataset del chatbot está en español. La comparación entre ambos modelos permitirá observar cuál recupera mejor el contexto correcto para responder preguntas relacionadas con el consultorio médico privado.

El modelo **`paraphrase-multilingual-MiniLM-L12-v2`** ya fue utilizado en el TP4, por lo que sirve como punto de comparación con el trabajo anterior. El modelo **`multilingual-e5-small`** se agrega como segunda alternativa para evaluar si mejora la recuperación semántica de las respuestas.



In [ ]:
# Modelo LLM elegido
modelo_llm = "google/flan-t5-small"

# Modelos de embeddings a comparar
modelo_embedding_1 = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo_embedding_2 = "intfloat/multilingual-e5-small"

print("Modelo LLM elegido:", modelo_llm)
print("Modelo de embedding 1:", modelo_embedding_1)
print("Modelo de embedding 2:", modelo_embedding_2)

Modelo LLM elegido: google/flan-t5-small
Modelo de embedding 1: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Modelo de embedding 2: intfloat/multilingual-e5-small


**c)** Implemente una clase ChatBot usando lo elegido en b).

Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD conocimiento es el dataset que Ud. planteó en el TP4.



**Respuesta:**

En esta sección se implementa una clase `ChatBot` utilizando **Chroma** como base de datos vectorial. El dataset original del TP4 se utiliza como base de conocimiento del chatbot.

Cada registro del dataset se transforma en un documento compuesto por la pregunta frecuente y su respuesta. Luego, esos documentos se convierten en embeddings y se almacenan en una colección de Chroma. Cuando el usuario realiza una consulta, el chatbot genera el embedding de la pregunta, busca los documentos más similares en la base vectorial y finalmente usa el modelo LLM elegido para redactar una respuesta basada en el contexto recuperado.

**Instalamos e importamos las librerias a utilizar**

In [ ]:
!pip install -q chromadb sentence-transformers transformers accelerate sentencepiece

In [ ]:
import pandas as pd
import torch
import chromadb

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
df["documento"] = (
    "Pregunta frecuente: " + df["pregunta"] +
    "\nRespuesta: " + df["respuesta"]
)

**Creamos ahora la clase Chatbot**

In [ ]:
class ChatBot:
    def __init__(
        self,
        df_conocimiento,
        modelo_embedding,
        modelo_llm,
        nombre_coleccion="chatbot_consultorio",
        top_k=2
    ):
        self.df = df_conocimiento.copy()
        self.modelo_embedding_nombre = modelo_embedding
        self.modelo_llm_nombre = modelo_llm
        self.nombre_coleccion = nombre_coleccion
        self.top_k = top_k

        # Cargar modelo de embeddings
        self.modelo_embedding = SentenceTransformer(modelo_embedding)

        # Documentos de conocimiento
        self.documentos = self.df["documento"].tolist()

        # IDs
        self.ids = [f"doc_{i}" for i in range(len(self.documentos))]

        # Metadatos
        self.metadatos = [
            {
                "pregunta_original": str(self.df.iloc[i]["pregunta"]),
                "respuesta_original": str(self.df.iloc[i]["respuesta"])
            }
            for i in range(len(self.df))
        ]

        # Preparar textos según el modelo de embeddings
        textos_documentos = self._preparar_textos_documentos(self.documentos)

        # Crear embeddings
        self.embeddings_documentos = self.modelo_embedding.encode(
            textos_documentos,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).tolist()

        # Crear cliente Chroma en memoria
        self.client = chromadb.EphemeralClient()

        # Si la colección ya existe, la borro para evitar errores al volver a ejecutar
        try:
            self.client.delete_collection(name=nombre_coleccion)
        except:
            pass

        # Crear colección usando similitud coseno
        self.collection = self.client.create_collection(
            name=nombre_coleccion,
            metadata={"hnsw:space": "cosine"}
        )

        # Agregar documentos a Chroma
        self.collection.add(
            ids=self.ids,
            documents=self.documentos,
            embeddings=self.embeddings_documentos,
            metadatas=self.metadatos
        )

        # Cargar modelo LLM
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.tokenizer = AutoTokenizer.from_pretrained(modelo_llm)
        self.llm = AutoModelForSeq2SeqLM.from_pretrained(modelo_llm).to(self.device)

    def _usa_e5(self):
        return "e5" in self.modelo_embedding_nombre.lower()

    def _preparar_textos_documentos(self, documentos):
        # E5 recomienda usar "passage:" para documentos
        if self._usa_e5():
            return ["passage: " + doc for doc in documentos]
        return documentos

    def _preparar_pregunta(self, pregunta):
        # E5 recomienda usar "query:" para preguntas
        if self._usa_e5():
            return "query: " + pregunta
        return pregunta

    def recuperar_contexto(self, pregunta):
        pregunta_preparada = self._preparar_pregunta(pregunta)

        embedding_pregunta = self.modelo_embedding.encode(
            [pregunta_preparada],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).tolist()

        resultados = self.collection.query(
            query_embeddings=embedding_pregunta,
            n_results=self.top_k
        )

        contextos = resultados["documents"][0]
        metadatos = resultados["metadatas"][0]
        distancias = resultados["distances"][0]

        return contextos, metadatos, distancias

    def responder(self, pregunta):
        contextos, metadatos, distancias = self.recuperar_contexto(pregunta)

        contexto = "\n\n".join(contextos)

        prompt = f"""
Use the context to answer the question.
Answer only in Spanish.
Do not invent information.

Context:
{contexto}

Question:
{pregunta}

Answer in Spanish:
"""

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        outputs = self.llm.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            num_beams=4
        )

        respuesta = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return respuesta.strip()

**d)** Pruebe el chatbot creado en c) con las preguntas de su dataset a)

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings.

Justifique y determine cuál elegiría para su aplicación.

**Cargamos el embedding MiniLM**

In [ ]:
chatbot_minilm = ChatBot(
    df_conocimiento=df,
    modelo_embedding=modelo_embedding_1,
    modelo_llm=modelo_llm,
    nombre_coleccion="consultorio_minilm",
    top_k=5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


**Cargamos el embedding e5**

In [ ]:
chatbot_e5 = ChatBot(
    df_conocimiento=df,
    modelo_embedding=modelo_embedding_2,
    modelo_llm=modelo_llm,
    nombre_coleccion="consultorio_e5",
    top_k=5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


**Evaluación de modelos:**

Para evaluar las respuestas usé una comparación flexible con **SequenceMatcher**. Primero normalicé los textos pasándolos a minúscula y quitando tildes. Luego calculé la similitud entre la **respuesta recuperada y la esperada**. Consideré como acierto los casos con una similitud igual o mayor al 70%, ya que no siempre las respuestas están escritas exactamente igual, aunque expresen la misma idea.

In [ ]:
from difflib import SequenceMatcher
import unicodedata

def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto

def similitud_texto(a, b):
    a = normalizar_texto(a)
    b = normalizar_texto(b)
    return SequenceMatcher(None, a, b).ratio()

In [ ]:
resultados = []

for _, fila in df_eval.iterrows():
    pregunta = fila["pregunta"]
    respuesta_esperada = fila["respuesta_esperada"]

    for nombre, bot in [("MiniLM", chatbot_minilm), ("E5", chatbot_e5)]:
        contextos, metadatos, distancias = bot.recuperar_contexto(pregunta)

        pregunta_recuperada = metadatos[0]["pregunta_original"]
        respuesta_recuperada = metadatos[0]["respuesta_original"]

        # Respuesta generada por el LLM usando el contexto recuperado
        respuesta_llm = bot.responder(pregunta)

        # Comparación flexible entre respuesta recuperada y respuesta esperada
        similitud = similitud_texto(respuesta_recuperada, respuesta_esperada)

        # Consideramos acierto si la similitud es mayor o igual a 0.70
        acierto_top1 = similitud >= 0.70

        resultados.append({
            "modelo_embedding": nombre,
            "pregunta_eval": pregunta,
            "respuesta_esperada": respuesta_esperada,
            "pregunta_recuperada_top1": pregunta_recuperada,
            "respuesta_recuperada_top1": respuesta_recuperada,
            "distancia_top1": distancias[0],
            "respuesta_llm": respuesta_llm,
            "similitud_respuesta": similitud,
            "acierto_top1": acierto_top1
        })

df_comparacion = pd.DataFrame(resultados)

df_comparacion

,modelo_embedding,pregunta_eval,respuesta_esperada,pregunta_recuperada_top1,respuesta_recuperada_top1,distancia_top1,respuesta_llm,similitud_respuesta,acierto_top1
0,MiniLM,Quiero reservar una consulta médica,Podés solicitar un turno escribiendo por Whats...,¿Puedo atenderme de forma particular?,"Sí, también se atienden pacientes particulares...",0.347900,Quiero reservar una consulta médica,0.170040,False
1,E5,Quiero reservar una consulta médica,Podés solicitar un turno escribiendo por Whats...,¿Puedo ir acompañado a la consulta?,"Sí, podés asistir acompañado si lo necesitás. ...",0.117422,Quiero reservar una consulta médica,0.308880,False
2,MiniLM,¿En qué horario atiende el consultorio?,El consultorio atiende de lunes a viernes de 9...,¿Cuáles son los horarios de atención?,El consultorio atiende de lunes a viernes de 9...,0.153926,El consultorio atiende de lunes a viernes de 9...,1.000000,True
3,E5,¿En qué horario atiende el consultorio?,El consultorio atiende de lunes a viernes de 9...,¿Cuáles son los horarios de atención?,El consultorio atiende de lunes a viernes de 9...,0.064724,lunes a viernes de 9:00 a 18:00 hs. Los horari...,1.000000,True
4,MiniLM,¿Me pueden pasar la ubicación del consultorio?,El consultorio se encuentra en Córdoba Capital...,¿Dónde queda el consultorio?,El consultorio se encuentra en Córdoba Capital...,0.363010,Respuesta: Se puede pasar la ubicación del con...,1.000000,True
5,E5,¿Me pueden pasar la ubicación del consultorio?,El consultorio se encuentra en Córdoba Capital...,¿Puedo ir acompañado a la consulta?,"Sí, podés asistir acompañado si lo necesitás. ...",0.134172,"Respuesta: S, puede asistir acompaado si lo ne...",0.253275,False
6,MiniLM,¿Trabajan con obras sociales?,"Sí, el consultorio trabaja con algunas obras s...",¿Atienden por obra social?,"Sí, el consultorio trabaja con algunas obras s...",0.333130,"Respuesta: S, el consultorio trabaja con algun...",1.000000,True
7,E5,¿Trabajan con obras sociales?,"Sí, el consultorio trabaja con algunas obras s...",¿Atienden por obra social?,"Sí, el consultorio trabaja con algunas obras s...",0.119700,"Respuesta: S, el consultorio trabaja con algun...",1.000000,True
8,MiniLM,¿Puedo pagar la consulta de manera particular?,"Sí, también se atienden pacientes particulares...",¿Cuáles son las formas de pago?,El consultorio acepta efectivo y transferencia...,0.313003,"Respuesta: S, también se atienden pacientes pa...",0.428571,False
9,E5,¿Puedo pagar la consulta de manera particular?,"Sí, también se atienden pacientes particulares...",¿Puedo atenderme de forma particular?,"Sí, también se atienden pacientes particulares...",0.111059,"Respuesta: S, también se atienden pacientes pa...",1.000000,True


**El chatbot sí funciona**

Porque ya hace el flujo completo:

pregunta → embedding → búsqueda en Chroma → recuperación de contexto → LLM → respuesta

O sea, **el sistema RAG está andando.**

**Veamos un resumen de como resulto la respuesta:**

In [ ]:
df_resumen = df_comparacion.groupby("modelo_embedding")[[
    "acierto_top1",
    "similitud_respuesta"
]].mean()

df_resumen

,acierto_top1,similitud_respuesta
modelo_embedding,,
E5,0.4,0.640991
MiniLM,0.4,0.644938


Para probar el chatbot, utilicé las preguntas del dataset de evaluación y comparé dos modelos de embeddings: `paraphrase-multilingual-MiniLM-L12-v2` y `intfloat/multilingual-e5-small`.

Primero evalué si la respuesta recuperada en el primer resultado coincidía con la respuesta esperada. Para evitar una comparación demasiado estricta, no utilicé igualdad exacta entre textos, sino una medida de similitud textual. Consideré como acierto aquellos casos donde la similitud entre la respuesta recuperada y la respuesta esperada fue mayor o igual a 0.70.

**Los resultados mostraron que ambos modelos obtuvieron un acierto top 1 de 0.4, es decir, un 40% de aciertos. En cuanto a la similitud promedio de las respuestas, MiniLM obtuvo aproximadamente 0.645 y E5 aproximadamente 0.641. Por lo tanto, ambos modelos tuvieron un rendimiento muy similar, aunque MiniLM mostró una leve ventaja en la similitud promedio.**

También observé que la calidad de la respuesta final generada por el LLM depende mucho del contexto recuperado. Cuando el embedding recupera una pregunta/respuesta adecuada, el chatbot puede generar una respuesta coherente. En cambio, cuando el contexto recuperado no corresponde bien con la consulta, el LLM tiende a dar respuestas poco útiles o repetir parte de la pregunta del usuario.

En conclusión, el chatbot funciona técnicamente porque realiza el proceso completo de recuperación de contexto y generación de respuesta. Sin embargo, **su desempeño todavía está limitado por la calidad de la recuperación semántica y por el tamaño reducido de la base de conocimiento.**


# e) BONUS (opcional)

Evalue el chatbot creado en c) para los modelos de embeddings elegidos y usados en d) usando el dataset a) con las métricas context precision y context recall. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas.

Dado sus resultados: ¿refuerza o no sus conclusiones realizadas en d) ?

Explique los resultados obtenidos y agregue sus conclusiones.

**Respuesta:**

En esta sección se evalúa el rendimiento de los modelos de embeddings usando las métricas **`context precision`** y **`context recall`**. **Estas métricas no miden directamente la calidad de redacción del LLM, sino qué tan bien el sistema recupera el contexto correcto desde la base de conocimiento**. Es decir, permiten analizar si los documentos recuperados son relevantes para responder la pregunta del usuario.


In [ ]:
chatbot_minilm = ChatBot(
    df_conocimiento=df,
    modelo_embedding=modelo_embedding_1,
    modelo_llm=modelo_llm,
    nombre_coleccion="consultorio_minilm",
    top_k=3
)

chatbot_e5 = ChatBot(
    df_conocimiento=df,
    modelo_embedding=modelo_embedding_2,
    modelo_llm=modelo_llm,
    nombre_coleccion="consultorio_e5",
    top_k=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
def evaluar_contextos(bot, df_eval, nombre_modelo, umbral_similitud=0.70):
    resultados = []

    for _, fila in df_eval.iterrows():
        pregunta = fila["pregunta"]
        respuesta_esperada = fila["respuesta_esperada"]

        contextos, metadatos, distancias = bot.recuperar_contexto(pregunta)

        respuestas_recuperadas = [
            metadata["respuesta_original"] for metadata in metadatos
        ]

        preguntas_recuperadas = [
            metadata["pregunta_original"] for metadata in metadatos
        ]

        # Evaluamos si cada contexto recuperado es relevante
        similitudes = [
            similitud_texto(respuesta_recuperada, respuesta_esperada)
            for respuesta_recuperada in respuestas_recuperadas
        ]

        relevantes = [
            similitud >= umbral_similitud
            for similitud in similitudes
        ]

        cantidad_relevantes = sum(relevantes)
        total_contextos = len(respuestas_recuperadas)

        # Métricas
        context_precision = cantidad_relevantes / total_contextos
        context_recall = 1 if cantidad_relevantes > 0 else 0

        # También guardamos si acertó en el primer resultado
        acierto_top1 = relevantes[0]

        resultados.append({
            "modelo_embedding": nombre_modelo,
            "pregunta_eval": pregunta,
            "respuesta_esperada": respuesta_esperada,
            "preguntas_recuperadas": preguntas_recuperadas,
            "respuestas_recuperadas": respuestas_recuperadas,
            "distancias": distancias,
            "similitudes": similitudes,
            "contextos_relevantes": cantidad_relevantes,
            "total_contextos": total_contextos,
            "context_precision": context_precision,
            "context_recall": context_recall,
            "acierto_top1": acierto_top1
        })

    return pd.DataFrame(resultados)

In [ ]:
metricas_minilm = evaluar_contextos(
    bot=chatbot_minilm,
    df_eval=df_eval,
    nombre_modelo="MiniLM",
    umbral_similitud=0.70
)

metricas_e5 = evaluar_contextos(
    bot=chatbot_e5,
    df_eval=df_eval,
    nombre_modelo="E5",
    umbral_similitud=0.70
)

df_metricas = pd.concat([metricas_minilm, metricas_e5], ignore_index=True)

df_metricas

,modelo_embedding,pregunta_eval,respuesta_esperada,preguntas_recuperadas,respuestas_recuperadas,distancias,similitudes,contextos_relevantes,total_contextos,context_precision,context_recall,acierto_top1
0,MiniLM,Quiero reservar una consulta médica,Podés solicitar un turno escribiendo por Whats...,"[¿Puedo atenderme de forma particular?, ¿Debo ...","[Sí, también se atienden pacientes particulare...","[0.34790003299713135, 0.4724280834197998, 0.50...","[0.1700404858299595, 0.24166666666666667, 0.35...",0,3,0.000000,0,False
1,MiniLM,¿En qué horario atiende el consultorio?,El consultorio atiende de lunes a viernes de 9...,"[¿Cuáles son los horarios de atención?, ¿Qué p...",[El consultorio atiende de lunes a viernes de ...,"[0.15392553806304932, 0.39211422204971313, 0.3...","[1.0, 0.3726235741444867, 0.4686192468619247]",1,3,0.333333,1,True
2,MiniLM,¿Me pueden pasar la ubicación del consultorio?,El consultorio se encuentra en Córdoba Capital...,"[¿Dónde queda el consultorio?, ¿Cómo confirmo ...",[El consultorio se encuentra en Córdoba Capita...,"[0.3630104660987854, 0.39121514558792114, 0.44...","[1.0, 0.22119815668202766, 0.25327510917030566]",1,3,0.333333,1,True
3,MiniLM,¿Trabajan con obras sociales?,"Sí, el consultorio trabaja con algunas obras s...","[¿Atienden por obra social?, ¿Qué pasa si lleg...","[Sí, el consultorio trabaja con algunas obras ...","[0.33313047885894775, 0.7507529854774475, 0.77...","[1.0, 0.30711610486891383, 0.38497652582159625]",1,3,0.333333,1,True
4,MiniLM,¿Puedo pagar la consulta de manera particular?,"Sí, también se atienden pacientes particulares...","[¿Cuáles son las formas de pago?, ¿Puedo atend...",[El consultorio acepta efectivo y transferenci...,"[0.31300318241119385, 0.3319384455680847, 0.40...","[0.42857142857142855, 1.0, 0.32653061224489793]",1,3,0.333333,1,False
5,MiniLM,Necesito cancelar mi turno,"Para cancelar un turno, escribí al WhatsApp de...","[¿Cómo hago para cancelar un turno?, ¿Qué pasa...","[Para cancelar un turno, escribí al WhatsApp d...","[0.33836615085601807, 0.585862398147583, 0.631...","[1.0, 0.3203125, 0.2948207171314741]",1,3,0.333333,1,True
6,MiniLM,Quiero cambiar el día de mi cita,"Sí, podés reprogramar tu turno escribiendo al ...","[¿Cuáles son los horarios de atención?, ¿Cómo ...",[El consultorio atiende de lunes a viernes de ...,"[0.5564086437225342, 0.6234351992607117, 0.647...","[0.39344262295081966, 0.4588744588744589, 0.31...",0,3,0.000000,0,False
7,MiniLM,¿Qué información tengo que enviar para pedir t...,"Para pedir un turno necesitás informar nombre,...","[¿Qué datos necesito para pedir un turno?, ¿Có...",[Para pedir un turno necesitás informar nombre...,"[0.3211895227432251, 0.44698262214660645, 0.49...","[1.0, 0.3114754098360656, 0.5714285714285714]",1,3,0.333333,1,True
8,MiniLM,"Tengo una emergencia, ¿puedo ir al consultorio?",El consultorio no atiende urgencias. En caso d...,"[¿Atienden urgencias?, ¿Puedo atenderme de for...",[El consultorio no atiende urgencias. En caso ...,"[0.38745129108428955, 0.5359840989112854, 0.58...","[1.0, 0.35797665369649806, 0.26459143968871596]",1,3,0.333333,1,True
9,MiniLM,¿Puedo pedir una receta por WhatsApp?,"Sí, podés solicitar recetas por WhatsApp si ya...","[¿Puedo pedir una receta por WhatsApp?, ¿Cómo ...",[Las recetas deben ser evaluadas por el profes...,"[0.24478554725646973, 0.3977290391921997, 0.49...","[0.5057471264367817, 0.5234375, 0.323943661971...",0,3,0.000000,0,False


# Veamos unas metricas para ver que modelo fue mejor:

* **`acierto_top1`**: mide si el **primer contexto recuperado** fue el correcto.
* **`context_precision`**: mide qué proporción de los contextos recuperados fueron realmente relevantes.
* **`context_recall`**: mide si el contexto correcto apareció **dentro de los contextos recuperados**.


In [ ]:
metricas_minilm = evaluar_contextos(
    bot=chatbot_minilm,
    df_eval=df_eval,
    nombre_modelo="MiniLM",
    umbral_similitud=0.70
)

metricas_e5 = evaluar_contextos(
    bot=chatbot_e5,
    df_eval=df_eval,
    nombre_modelo="E5",
    umbral_similitud=0.70
)

df_metricas = pd.concat([metricas_minilm, metricas_e5], ignore_index=True)

df_resumen_bonus = df_metricas.groupby("modelo_embedding")[[
    "acierto_top1",
    "context_precision",
    "context_recall"
]].mean()

df_resumen_bonus

,acierto_top1,context_precision,context_recall
modelo_embedding,,,
E5,0.4,0.166667,0.50
MiniLM,0.4,0.150000,0.45


Para el cálculo de las métricas se utilizó **top_k=3**, ya que el dataset es reducido y recuperar demasiados contextos podía agregar ruido. Con este valor, se evaluó si los contextos recuperados eran relevantes para responder cada pregunta del dataset de evaluación.

Los resultados muestran que ambos modelos obtuvieron el mismo acierto top 1, con un valor de 0.40. Sin embargo, E5 obtuvo un context precision de 0.1667 y un context recall de 0.50, mientras que MiniLM obtuvo un context precision de 0.15 y un context recall de 0.45.

Esto indica que, aunque ambos modelos tuvieron un rendimiento parecido, **E5 logró recuperar contexto relevante en una proporción ligeramente mayor.** Por lo tanto, estos resultados refuerzan parcialmente lo observado en el punto d), donde ambos embeddings se comportaban de manera similar, pero en esta evaluación métrica **E5 mostró una leve ventaja**.

**En conclusión, el chatbot logra recuperar información útil desde la base de conocimiento, pero su rendimiento todavía es limitado. La calidad de las respuestas depende mucho de que el embedding recupere el contexto correcto. Para mejorar el sistema, sería conveniente ampliar la base de conocimiento, agregar más variantes de preguntas y seguir probando **otros** modelos de embeddings.**

# f) BONUS (opcional)

Evalue el modelo LLM usado para el chatbot creado en c) con el modelo de embedding que mejor le haya resultado en d) usando el dataset a).

Use las métricas Answer Relevancy  y Faithfulness. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas. Explique los resultados obtenidos y agregue sus conclusiones.

# Introducción:

En esta sección evalúo la calidad de las respuestas generadas por el LLM usando el chatbot con el embedding E5, que fue el que mostró una leve ventaja en la recuperación de contexto. **A diferencia del bonus anterior, acá no se evalúa solamente si el contexto fue recuperado correctamente, sino si la respuesta generada es relevante y si está fundamentada en el contexto recuperado.**

**Answer Relevancy:** mide qué tan parecida/relevante es la respuesta generada respecto a la respuesta esperada.

**Faithfulness:** mide qué tan apoyada está la respuesta generada en los contextos recuperados

In [ ]:
chatbot_e5 = ChatBot(
    df_conocimiento=df,
    modelo_embedding=modelo_embedding_2,
    modelo_llm=modelo_llm,
    nombre_coleccion="consultorio_e5",
    top_k=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
import numpy as np

def limpiar_respuesta_llm(texto):
    texto = str(texto).strip()

    prefijos = [
        "Respuesta:",
        "Answer:",
        "Spanish answer:"
    ]

    for prefijo in prefijos:
        if texto.lower().startswith(prefijo.lower()):
            texto = texto[len(prefijo):].strip()

    return texto


def similitud_embedding(modelo_embedding, texto_1, texto_2):
    embeddings = modelo_embedding.encode(
        [
            "passage: " + str(texto_1),
            "passage: " + str(texto_2)
        ],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return float(np.dot(embeddings[0], embeddings[1]))

In [ ]:
resultados_llm = []

for _, fila in df_eval.iterrows():
    pregunta = fila["pregunta"]
    respuesta_esperada = fila["respuesta_esperada"]

    # Recuperamos contexto con E5
    contextos, metadatos, distancias = chatbot_e5.recuperar_contexto(pregunta)

    # Generamos respuesta con el LLM
    respuesta_llm = chatbot_e5.responder(pregunta)
    respuesta_llm_limpia = limpiar_respuesta_llm(respuesta_llm)

    # Answer Relevancy:
    # similitud entre respuesta generada y respuesta esperada
    answer_relevancy = similitud_embedding(
        chatbot_e5.modelo_embedding,
        respuesta_llm_limpia,
        respuesta_esperada
    )

    # Faithfulness:
    # similitud máxima entre la respuesta generada y los contextos recuperados
    similitudes_contexto = [
        similitud_embedding(
            chatbot_e5.modelo_embedding,
            respuesta_llm_limpia,
            contexto
        )
        for contexto in contextos
    ]

    faithfulness = max(similitudes_contexto)

    resultados_llm.append({
        "pregunta_eval": pregunta,
        "respuesta_esperada": respuesta_esperada,
        "respuesta_llm": respuesta_llm_limpia,
        "contextos_recuperados": contextos,
        "answer_relevancy": answer_relevancy,
        "faithfulness": faithfulness
    })

df_eval_llm = pd.DataFrame(resultados_llm)

df_eval_llm

,pregunta_eval,respuesta_esperada,respuesta_llm,contextos_recuperados,answer_relevancy,faithfulness
0,Quiero reservar una consulta médica,Podés solicitar un turno escribiendo por Whats...,Quiero reservar una consulta médica,[Pregunta frecuente: ¿Puedo ir acompañado a la...,0.878588,0.859782
1,¿En qué horario atiende el consultorio?,El consultorio atiende de lunes a viernes de 9...,El consultorio encuentra en Córdoba Capital. L...,[Pregunta frecuente: ¿Cuáles son los horarios ...,0.915070,0.945885
2,¿Me pueden pasar la ubicación del consultorio?,El consultorio se encuentra en Córdoba Capital...,"S, puede asistir acompaado si lo necesitás. En...",[Pregunta frecuente: ¿Puedo ir acompañado a la...,0.869171,0.926089
3,¿Trabajan con obras sociales?,"Sí, el consultorio trabaja con algunas obras s...","S, el consultorio trabaja con algunas obras so...",[Pregunta frecuente: ¿Atienden por obra social...,0.965211,0.963928
4,¿Puedo pagar la consulta de manera particular?,"Sí, también se atienden pacientes particulares...","S, también se atienden pacientes particulares....",[Pregunta frecuente: ¿Puedo atenderme de forma...,0.933909,0.955101
5,Necesito cancelar mi turno,"Para cancelar un turno, escribí al WhatsApp de...",Respuesta frecuente: Cómo hago para cancelar u...,[Pregunta frecuente: ¿Cómo hago para cancelar ...,0.943788,0.989751
6,Quiero cambiar el día de mi cita,"Sí, podés reprogramar tu turno escribiendo al ...",Quiero cambiar el da de mi cita,[Pregunta frecuente: ¿Cómo confirmo mi turno?\...,0.839696,0.836478
7,¿Qué información tengo que enviar para pedir t...,"Para pedir un turno necesitás informar nombre,...",Para pedir una turno necesitás informar nombre...,[Pregunta frecuente: ¿Qué datos necesito para ...,0.961006,0.987697
8,"Tengo una emergencia, ¿puedo ir al consultorio?",El consultorio no atiende urgencias. En caso d...,"S, puede asistir acompaado si lo necesitás. En...",[Pregunta frecuente: ¿Atienden urgencias?\nRes...,0.847982,0.901990
9,¿Puedo pedir una receta por WhatsApp?,"Sí, podés solicitar recetas por WhatsApp si ya...",Los recetas deben ser evaluadas por el profesi...,[Pregunta frecuente: ¿Puedo pedir una receta p...,0.970131,0.957092


In [ ]:
df_resumen_llm = df_eval_llm[[
    "answer_relevancy",
    "faithfulness"
]].mean().to_frame(name="promedio")

df_resumen_llm

,promedio
answer_relevancy,0.913275
faithfulness,0.938528


Al evaluar el LLM usando el modelo de embedding E5, se obtuvieron valores promedio de 0.913 en Answer Relevancy y 0.939 en Faithfulness. **Estos resultados indican que, en general, las respuestas generadas por el modelo fueron relevantes respecto a las respuestas esperadas y estuvieron bastante apoyadas en los contextos recuperados.**

Sin embargo, estos valores deben interpretarse con cautela, ya que fueron calculados mediante similitud de embeddings. Esto permite medir cercanía semántica entre textos, pero no reemplaza completamente una evaluación humana de la calidad de la respuesta. En algunos casos, el modelo generó respuestas incompletas o poco naturales, aunque semánticamente relacionadas con el contexto.

En conclusión, el LLM logra generar respuestas relacionadas con la información recuperada, pero su desempeño depende fuertemente de que el contexto recuperado sea correcto. Los resultados muestran un funcionamiento aceptable del esquema RAG, aunque todavía sería necesario mejorar la base de conocimiento, ampliar las variantes de preguntas y probar modelos LLM más potentes para obtener respuestas más precisas y naturales.


# Conclusión final:

En conclusión, el chatbot implementado logra funcionar correctamente como un sistema RAG, ya que realiza el circuito completo: recibe una consulta del usuario, transforma esa consulta en un embedding, recupera contexto desde la base vectorial y luego utiliza un LLM para generar una respuesta.

A partir de las pruebas realizadas, se observó que la calidad final del chatbot no depende solamente del modelo LLM, sino principalmente de la calidad de la base de conocimiento y de la recuperación del contexto. Cuando el sistema recupera información relevante, el LLM puede generar respuestas más coherentes y útiles. En cambio, cuando el contexto recuperado no corresponde bien con la consulta, la respuesta final resulta incompleta, poco precisa o directamente incorrecta.

En la comparación de embeddings, los modelos MiniLM y E5 tuvieron un rendimiento similar, aunque **E5 mostró una leve ventaj**a en las métricas de recuperación al usar `top_k=3`. Esto indica que el modelo de embedding influye directamente en la calidad del contexto que recibe el LLM.

**También se observó que las métricas del LLM, como Answer Relevancy y Faithfulness, dieron valores alto**s. Sin embargo, estos resultados deben interpretarse con cuidado, ya que el LLM puede generar respuestas semánticamente relacionadas con el contexto, pero eso no garantiza que la recuperación inicial haya sido siempre la mejor.

Por eso, **antes de intentar mejorar el chatbot usando un LLM más grande o más pesado, sería más eficiente ampliar y mejorar la base de conocimiento. Agregar más preguntas frecuentes, variantes de consultas y respuestas mejor estructuradas permitiría que el sistema recupere mejor información relevante.** Una vez mejorada esa base documental, recién tendría más sentido probar modelos LLM más potentes para mejorar la redacción y naturalidad de las respuestas.


# g) REFERENCIAS.

Hugging Face. (s.f.). google/flan-t5-small. Hugging Face Model Card.

Hugging Face. (s.f.). FLAN-T5. Transformers documentation.

Hugging Face. (s.f.). Auto Classes. Transformers documentation.

Sentence Transformers. (s.f.). paraphrase-multilingual-MiniLM-L12-v2. Hugging Face Model Card.

Wang, L., Yang, N., Huang, X., Yang, L., Majumder, R., & Wei, F. (2024). Multilingual E5 Text Embeddings: A Technical Report. arXiv.

Hugging Face. (s.f.). intfloat/multilingual-e5-small. Hugging Face Model Card.

Chroma. (s.f.). Configure Collections. Chroma Docs.

Chroma. (s.f.). Query and Get. Chroma Docs.

RAGAS. (s.f.). Metrics: Faithfulness, Answer Relevance, Context Precision and Context Recall. RAGAS documentation.

Python Software Foundation. (s.f.). difflib — Helpers for computing deltas. Python documentation.

Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. arXiv.
